In [ ]:
%pip install -q kailash-ml polars plotly gdown python-dotenv


# ════════════════════════════════════════════════════════════════════════
# ASCENT07 — Exercise 8: Capstone — End-to-End Deep Learning Pipeline
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Complete DL pipeline from data to deployment: TrainingPipeline
#   → ModelRegistry → OnnxBridge → InferenceServer.
#
# TASKS:
#   1. Load and preprocess image data
#   2. Train CNN via TrainingPipeline
#   3. Register in ModelRegistry with metrics
#   4. Export to ONNX via OnnxBridge
#   5. Deploy via InferenceServer and test predictions
#   6. Compare ONNX inference speed vs original
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import pickle
import time

import polars as pl

from kailash.infrastructure import ConnectionManager
from kailash_ml import (
    InferenceServer,
    ModelRegistry,
    ModelVisualizer,
    OnnxBridge,
    TrainingPipeline,
)
from kailash_ml.types import MetricSpec

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()



## TASK 1: Load and preprocess image data


In [ ]:
loader = ASCENTDataLoader()
data = loader.load("ascent07", "fashion_mnist_sample.parquet")

pixel_cols = [c for c in data.columns if c != "label"]
n_samples = data.height

# Normalize pixel values to [0, 1]
normalized = data.with_columns([(pl.col(c) / 255.0).alias(c) for c in pixel_cols])

# Train/test split (80/20)
n_train = int(n_samples * 0.8)
train_data = normalized[:n_train]
test_data = normalized[n_train:]

print(f"=== Fashion-MNIST Pipeline ===")
print(f"Total: {n_samples}, Train: {n_train}, Test: {n_samples - n_train}")
print(f"Features: {len(pixel_cols)} pixels (28×28 flattened)")
print(f"Classes: 10 (T-shirt, Trouser, Pullover, Dress, Coat,")
print(f"          Sandal, Shirt, Sneaker, Bag, Ankle boot)")



## TASK 2: Train CNN via TrainingPipeline


In [ ]:
# TODO: Create a TrainingPipeline for neural network classification.
# Hint: TrainingPipeline(
#     model_type="neural_network",
#     target="label",
#     features=pixel_cols,
#     config={
#         "architecture": "cnn",
#         "hidden_layers": [128, 64],
#         "activation": "relu",
#         "dropout": 0.3,
#         "epochs": 10,
#         "batch_size": 32,
#         "learning_rate": 0.001,
#         "optimizer": "adam",
#     },
# )
pipeline = ____

print(f"\n=== Training via TrainingPipeline ===")
start_time = time.time()

# TODO: Fit the pipeline on training data.
# Hint: pipeline.fit(train_data)
result = ____
train_time = time.time() - start_time

print(f"Training time: {train_time:.1f}s")
print(f"Final training loss: {result.metrics.get('loss', 'N/A')}")
print(f"Training accuracy: {result.metrics.get('accuracy', 'N/A')}")

# TODO: Generate predictions on test data.
# Hint: pipeline.predict(test_data)
predictions = ____
test_labels = test_data["label"].to_list()
pred_labels = predictions["prediction"].to_list()

correct = sum(1 for p, t in zip(pred_labels, test_labels) if p == t)
test_accuracy = correct / len(test_labels)
print(f"Test accuracy: {test_accuracy:.4f}")

# Visualize training curves
viz = ModelVisualizer()
fig = viz.plot_training_curves(result.history)
fig.write_html("capstone_training_curves.html")
print(f"Training curves saved to capstone_training_curves.html")



## TASK 3: Register in ModelRegistry


In [ ]:
async def register_model():
    conn = ConnectionManager("sqlite:///capstone_models.db")
    await conn.initialize()

    registry = ModelRegistry(conn)
    await registry.initialize()

    # TODO: Register the model with name, artifact, and metrics.
    # Hint: registry.register_model(
    #     name="fashion_mnist_cnn",
    #     artifact=pickle.dumps(result.model),
    #     metrics=[
    #         MetricSpec(name="test_accuracy", value=test_accuracy),
    #         MetricSpec(name="train_time_seconds", value=train_time),
    #         MetricSpec(name="parameters", value=result.metrics.get("n_params", 0)),
    #     ],
    # )
    version = await ____

    # TODO: Promote the model to production stage.
    # Hint: registry.promote_model(name="fashion_mnist_cnn", version=version.version, target_stage="production")
    await ____

    print(f"\n=== ModelRegistry ===")
    print(f"Registered: fashion_mnist_cnn v{version.version}")
    print(f"Stage: production")
    print(f"Metrics: accuracy={test_accuracy:.4f}, train_time={train_time:.1f}s")

    return registry, version


registry, model_version  = await register_model()



## TASK 4: Export to ONNX via OnnxBridge


In [ ]:
async def export_onnx():
    bridge = OnnxBridge()

    # TODO: Export the trained model to ONNX format.
    # Hint: bridge.export(model=result.model, input_shape=(1, len(pixel_cols)), output_path="fashion_mnist_cnn.onnx")
    onnx_path = ____

    # Validate ONNX output matches original model
    test_sample = test_data.select(pixel_cols).row(0)
    # TODO: Validate the ONNX model against expected output.
    # Hint: bridge.validate(onnx_path, test_data=[list(test_sample)], expected=[pred_labels[:1]])
    metrics = ____

    print(f"\n=== ONNX Export ===")
    print(f"Path: {onnx_path}")
    print(f"Validation: {metrics}")
    print(f"ONNX is platform-agnostic: deploy to mobile, edge, browser, or server")

    return bridge, onnx_path


bridge, onnx_path  = await export_onnx()



## TASK 5: Deploy via InferenceServer


In [ ]:
async def deploy_and_test():
    # TODO: Create an InferenceServer with the ONNX model.
    # Hint: InferenceServer(model_path=onnx_path, port=8090)
    server = ____

    print(f"\n=== InferenceServer Deployment ===")
    await server.start()
    print(f"Server running on port 8090")

    # Test predictions
    class_names = [
        "T-shirt",
        "Trouser",
        "Pullover",
        "Dress",
        "Coat",
        "Sandal",
        "Shirt",
        "Sneaker",
        "Bag",
        "Ankle boot",
    ]

    for i in range(3):
        sample = list(test_data.select(pixel_cols).row(i))
        # TODO: Run a prediction through the inference server.
        # Hint: await server.predict(sample)
        prediction = ____
        true_label = int(test_data["label"][i])
        pred_class = prediction.get("class", prediction.get("prediction", 0))
        print(
            f"  Sample {i+1}: true={class_names[true_label]}, "
            f"pred={class_names[int(pred_class)]}"
        )

    await server.stop()
    print(f"Server stopped.")

    return server


server  = await deploy_and_test()



## TASK 6: Compare inference speed


In [ ]:
print(f"\n=== Inference Speed Comparison ===")

# Original model inference
n_test = 100
test_samples = [list(test_data.select(pixel_cols).row(i)) for i in range(n_test)]

start = time.time()
for sample in test_samples:
    pipeline.predict(
        pl.DataFrame({"label": [0], **{c: [v] for c, v in zip(pixel_cols, sample)}})
    )
original_time = time.time() - start

print(
    f"Original model: {n_test} predictions in {original_time:.3f}s "
    f"({original_time/n_test*1000:.1f}ms/prediction)"
)
print(f"ONNX model: typically 2-5× faster due to graph optimizations")
print(f"\nONNX advantages:")
print(f"  - Graph-level optimizations (operator fusion, constant folding)")
print(f"  - Platform-native execution (CPU vectorization, GPU kernels)")
print(f"  - No Python overhead at inference time")
print(f"  - Single file deployment (model + weights in one .onnx)")

print(f"\n=== Full Pipeline Summary ===")
print(f"1. Data: {n_samples} Fashion-MNIST images → normalized")
print(f"2. Training: TrainingPipeline (CNN, Adam, dropout=0.3)")
print(f"3. Registry: ModelRegistry (versioned, promoted to production)")
print(f"4. Export: OnnxBridge (validated, portable)")
print(f"5. Deploy: InferenceServer (HTTP endpoint, batch support)")
print(f"This is the Kailash DL lifecycle — from pixels to production.")

print("\n✓ Exercise 8 complete — end-to-end DL pipeline with Kailash")

